# Pneumonia Detection Notebook

Este notebook converte o projeto `pneumonia-detection` em um fluxo interativo de treinamento, avaliação e Grad-CAM.

## Configuração

Ajuste `base_dir` se necessário e abra este notebook em Jupyter ou VS Code.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import cv2

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix

base_dir = 'data/chest_xray'
models_dir = 'models'
outputs_dir = 'outputs'
os.makedirs(models_dir, exist_ok=True)
os.makedirs(outputs_dir, exist_ok=True)


In [ ]:
def get_data_generators(base_dir):
    train_dir = os.path.join(base_dir, 'train')
    val_dir = os.path.join(base_dir, 'val')
    test_dir = os.path.join(base_dir, 'test')

    train_datagen = ImageDataGenerator(rescale=1./255)
    test_datagen = ImageDataGenerator(rescale=1./255)

    train_data = train_datagen.flow_from_directory(
        train_dir,
        target_size=(224,224),
        batch_size=32,
        class_mode='binary'
    )

    val_data = test_datagen.flow_from_directory(
        val_dir,
        target_size=(224,224),
        batch_size=32,
        class_mode='binary'
    )

    test_data = test_datagen.flow_from_directory(
        test_dir,
        target_size=(224,224),
        batch_size=32,
        class_mode='binary',
        shuffle=False
    )

    return train_data, val_data, test_data

In [ ]:
def build_model():
    base_model = DenseNet121(
        weights='imagenet',
        include_top=False,
        input_shape=(224,224,3)
    )
    base_model.trainable = False

    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(128, activation='relu')(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs=base_model.input, outputs=output)
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
train_data, val_data, test_data = get_data_generators(base_dir)
model = build_model()
model.summary()

## Treinamento

Execute esta célula para treinar o modelo. O treinamento pode demorar dependendo do hardware.

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)
model.save(os.path.join(models_dir, 'model.h5'))

## Avaliação

Carregue o modelo salvo e avalie no conjunto de teste.

In [ ]:
model = load_model(os.path.join(models_dir, 'model.h5'))
_, _, test_data = get_data_generators(base_dir)
preds = model.predict(test_data)
preds = (preds > 0.5).astype(int).reshape(-1)
print(classification_report(test_data.classes, preds))

cm = confusion_matrix(test_data.classes, preds)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

## Grad-CAM

Visualize a ativação do modelo sobre uma imagem de teste.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output] 
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_channel = predictions[:, 0]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
img_path = os.path.join(base_dir, 'test', 'NORMAL', 'NORMAL2-IM-0007-0001.jpeg')
img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224,224))
img_array = tf.keras.preprocessing.image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0) / 255.0

heatmap = make_gradcam_heatmap(model, img_array, 'conv5_block16_concat')

img = cv2.imread(img_path)
img = cv2.resize(img, (224,224))
heatmap = cv2.resize(heatmap, (224,224))
heatmap = np.uint8(255 * heatmap)
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
superimposed = heatmap * 0.4 + img

plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(superimposed.astype('uint8'), cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

cv2.imwrite(os.path.join(outputs_dir, 'gradcam_example.png'), superimposed)